In [ ]:
import pandas as pd

from pandas import ExcelWriter

from time import sleep

import datetime

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager

import os

import zipfile


import ssl
ssl._create_default_https_context = ssl._create_unverified_context
from DrissionPage import ChromiumPage, ChromiumOptions


#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'US FFIEC' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

#writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder
#driver = webdriver.Chrome(options=chromeOptions)

options = ChromiumOptions()
options.set_download_path(tempfolder)
driver = ChromiumPage(options)
#------------------------------------------------ Begin_Variable ----------------------------------------

# regdict= {'FCA':'1', 'FDIC':'2', 'nan':'3', 'FRS':'4', 'NCUA':'5', 'OCC':'6'}
regdict= {'FCA':'1', 'FDIC':'2', 'FHFA':'3', 'FRS':'4', 'NCUA':'5', 'OCC':'6','nan':'7'}


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

		  'Phone - Mother company': [], 'Check': []}



ISO= {"AFGHANISTAN": "AF", "ÅLAND ISLANDS": "AX", "ALBANIA": "AL", "ALGERIA": "DZ", "AMERICAN SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARCTICA": "AQ", "ANTIGUA AND BARBUDA": "AG", "ARGENTINA": "AR", "ARMENIA": "AM", "ARUBA": "AW", "AUSTRALIA": "AU", "AUSTRIA": "AT", "AZERBAIJAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESH": "BD", "BARBADOS": "BB", "BELARUS": "BY", "BELGIUM": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIA": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIA AND HERZEGOVINA": "BA", "BOTSWANA": "BW", "BOUVET ISLAND": "BV", "BRAZIL": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIA": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "CABO VERDE": "CV", "CAMBODIA": "KH", "CAMEROON, UNITED REPUBLIC OF": "CM", "CANADA": "CA", "CAYMAN ISLANDS": "KY", "CENTRAL AFRICAN REPUBLIC": "CF", "CHAD": "TD", "CHILE": "CL", "CHINA, PEOPLES REPUBLIC OF": "CN", "CHRISTMAS ISLAND": "CX", "COCOS (KEELING) ISLANDS": "CC", "COLOMBIA": "CO", "COMOROS": "KM", "CONGO": "CG", "CONGO, DEMOCRATIC REPUBLIC OF THE": "CD", "COOK ISLANDS": "CK", "COSTA RICA": "CR", "CÔTE D'IVOIRE": "CI", "CROATIA": "HR", "CUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "CYPRUS": "CY", "CZECH REPUBLIC": "CZ", "DENMARK": "DK", "DJIBOUTI": "DJ", "DOMINICA": "DM", "DOMINICAN REPUBLIC": "DO", "ECUADOR": "EC", "EGYPT": "EG", "EL SALVADOR": "SV", "EQUATORIAL GUINEA": "GQ", "ERITREA": "ER", "ESTONIA": "EE", "ESWATINI": "SZ", "ETHIOPIA": "ET", "FALKLAND ISLANDS (MALVINAS)": "FK", "FAROE ISLANDS": "FO", "FIJI": "FJ", "FINLAND": "FI", "FRANCE": "FR", "FRENCH GUIANA": "GF", "FRENCH POLYNESIA": "PF", "FRENCH SOUTHERN TERRITORIES": "TF", "GABON": "GA", "GAMBIA": "GM", "GEORGIA": "GE", "GERMANY": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GREECE": "GR", "GREENLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "HUNGARY": "HU", "ICELAND": "IS", "INDIA": "IN", "INDONESIA": "ID", "IRAN": "IR", "IRAQ": "IQ", "IRELAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALY": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDAN": "JO", "KAZAKHSTAN": "KZ", "KENYA": "KE", "KIRIBATI": "KI", """KOREA (DEMOCRATIC PEOPLE"S REPUBLIC OF)""": "KP", "KOREA, SOUTH": "KR", "KUWAIT": "KW", "KYRGYZSTAN": "KG", "LAO PEOPLE'S DEMOCRATIC REPUBLIC": "LA", "LATVIA": "LV", "LEBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYA": "LY", "LIECHTENSTEIN": "LI", "LITHUANIA": "LT", "LUXEMBOURG": "LU", "MACAU": "MO", "MADAGASCAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALDIVES": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALL ISLANDS": "MH", "MARTINIQUE": "MQ", "MAURITANIA": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FEDERATED STATES OF MICRONESIA": "FM", "MOLDOVA, REPUBLIC OF": "MD", "MONACO": "MC", "MONGOLIA": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MOROCCO": "MA", "MOZAMBIQUE": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NETHERLANDS": "NL", "NEW CALEDONIA": "NC", "NEW ZEALAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORFOLK ISLAND": "NF", "NORTH MACEDONIA": "MK", "NORTHERN MARIANA ISLANDS": "MP", "NORWAY": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "PALESTINE, STATE OF": "PS", "PANAMA": "PA", "PAPUA NEW GUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINES": "PH", "PITCAIRN": "PN", "POLAND": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "QATAR": "QA", "RÉUNION": "RE", "ROMANIA": "RO", "RUSSIA": "RU", "RWANDA": "RW", "SAINT BARTHÉLEMY": "BL", "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA": "SH", "SAINT KITTS AND NEVIS": "KN", "SAINT LUCIA": "LC", "SAINT MARTIN (FRENCH PART)": "MF", "SAINT PIERRE AND MIQUELON": "PM", "SAINT VINCENT AND THE GRENADINES": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME AND PRINCIPE": "ST", "SAUDI ARABIA": "SA", "SENEGAL": "SN", "SERBIA": "RS", "SEYCHELLES": "SC", "SIERRA LEONE": "SL", "SINGAPORE": "SG", "SINT MAARTEN (DUTCH PART)": "SX", "SLOVAKIA": "SK", "SLOVENIA": "SI", "SOLOMON ISLANDS": "SB", "SOMALIA": "SO", "SOUTH AFRICA": "ZA", "SOUTH GEORGIA AND THE SOUTH SANDWICH ISLANDS": "GS", "SOUTH SUDAN": "SS", "SPAIN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SVALBARD AND JAN MAYEN": "SJ", "SWEDEN": "SE", "SWITZERLAND": "CH", "SYRIAN ARAB REPUBLIC": "SY", "TAIWAN": "TW", "TAJIKISTAN": "TJ", "TANZANIA, UNITED REPUBLIC OF": "TZ", "THAILAND": "TH", "TIMOR-LESTE": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD AND TOBAGO": "TT", "TUNISIA": "TN", "TURKEY": "TR", "TURKMENISTAN": "TM", "TURKS & CAICOS ISLANDS": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "UNITED ARAB EMIRATES": "AE", "UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND": "GB", "UNITED STATES": "US", "UNITED STATES MINOR OUTLYING ISLANDS": "UM", "URUGUAY": "UY", "UZBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISH VIRGIN ISLANDS": "VG", "VIRGIN ISLANDS OF THE U.S.": "VI", "WALLIS AND FUTUNA": "WF", "WESTERN SAHARA": "EH", "YEMEN": "YE", "ZAMBIA": "ZM", "ZIMBABWE": "ZW", "ENGLAND": "GB", "UNITED KINGDOM (OTHER)": "GB", "FRANCE (OTHER)": "FR", "WALES": "GB", "CONGO (KINSHASA)": "CD", "CONGO (BRAZZAVILLE)": "CD", "SCOTLAND": "GB", "ITALY (OTHER)": "IT", "INDONESIA (OTHER)": "ID", "INDIA (OTHER)": "IN", "MOROCCO (OTHER)": "MA", "NEW ZEALAND (OTHER)": "NZ", "SWITZERLAND (OTHER)": "CH", "MALAYSIA (OTHER)": "MY", "NETHERLANDS ANTILLES": "AN", "TRINIDAD & TOBAGO (OTHER)": "TT", "CHANNEL ISLANDS": "GB", "UNITED ARAB EMIRATES (OTHER)": "AE", "DENMARK (OTHER)": "DK", "COMORO ISLANDS": "KM", "MACEDONIA (FORMER YUGOSLAV REPUBLIC OF)": "MK", "SERBIA AND MONTENEGRO(FORMER YUGOSLAVIA)": "CS", "TRINIDAD": "TT", "ETHIOPIA (OTHER)": "ET", "IVORY COAST": "CI", "DUBAI": "AE", "BRITISH WEST INDIES (OTHER)": "VG", "SWAZILAND": "SZ", 'UNITED KINGDOM  (OTHER)': 'GB'}

processdate = now.strftime('%Y-%m-%d')

mapping = {
    1: "Primary Federal Regulator: Farm Credit Administration",
    2: "Primary Federal Regulator: Federal Deposit Insurance Corporation",
    3: "Primary Federal Regulator: Federal Housing Finance Agency",
    4: "Primary Federal Regulator: Federal Reserve System",
    5: "Primary Federal Regulator: National Credit Union Administration",
    6: "Primary Federal Regulator: Office of the Comptroller of the Currency",
    7: "Primary Federal Regulator: Non-bank Companies"
}

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://www.ffiec.gov/npw/FinancialReport/DataDownload')

sleep(4)
download_button = driver.ele('x://button[@onclick="ReturnAttributesActiveZipFileCSV()"]')

download_button.click.to_download(save_path=tempfolder)

sleep(15)


# tempfile = []
# if not tempfile:
#     tempfile = [f for f in os.listdir(tempfolder) if '.tmp' not in f and 'crdownload' not in f]
#     sleep(1)
# else:
#     download_botton = driver.find_element(By.XPATH, "//button[@onclick='ReturnAttributesActiveZipFileCSV()']")
#     sleep(3)
#     download_botton.click()
#     sleep(15)
#     tempfile = [f for f in os.listdir(tempfolder) if '.tmp' not in f and 'crdownload' not in f]
#     sleep(1)
tempfile = [f for f in os.listdir(tempfolder) if '.tmp' not in f and 'crdownload' not in f]
download_path = os.path.join(tempfolder, tempfile[0])


fsize = os.path.getsize(download_path)
sleep(3)
while True:
    new_size = os.path.getsize(download_path)
    if new_size <= fsize:
        break
    fsize = new_size
    sleep(0.5)

tempfile = os.listdir(tempfolder)
if len(tempfile) > 1:
    print('Error! There is more than one file in tempfolder.')

with zipfile.ZipFile(download_path) as zip_file:
    zip_file.extractall(tempfolder)

os.remove(download_path)
# find CSV file(s) in tempfolder (case-insensitive)
csv_files = [f for f in os.listdir(tempfolder) if f.lower().endswith('.csv')]
if not csv_files:
	raise FileNotFoundError(f"No CSV file found in {tempfolder} after extracting Zip")
# use full path for reading
inputfile = os.path.join(tempfolder, csv_files[0])
df = pd.read_csv(inputfile, delimiter=',', dtype=str)



#####CSV SQL ready file dict starts here######

df.columns=df.columns.str.strip()

df.columns=map(str.upper, df.columns)



for col in df.columns:

	df[col]= list(map(lambda x: str(x).strip(), df[col].tolist()))

	df[col]= list(map(lambda x: '' if x=='0' else x, df[col].tolist()))



sqldict['Name']=df['NM_LGL'].tolist()

reglenght=len(sqldict['Name'])

sqldict['City']=df['CITY'].tolist()

sqldict['Address_1']=df['STREET_LINE1'].tolist()

sqldict['InternalID_1']=df['#ID_RSSD'].tolist()

sqldict['InternalID_2']=df['ID_FDIC_CERT'].tolist()

sqldict['InternalID_3']=df['NM_SRCH_CD'].tolist()

sqldict['Typology']=df['ENTITY_TYPE'].tolist()

sqldict['Website']=df['URL'].tolist()



sqldict['LEI Code']=df['ID_LEI'].tolist()



PRIM_FED_REG=df['PRIM_FED_REG'].tolist()

STATE_ABBR_NM=df['STATE_ABBR_NM'].tolist()

ZIP_CD=df['ZIP_CD'].tolist()

CNTRY_NM=df['CNTRY_NM'].tolist()

for times in range(reglenght):

	try:

		sqldict['Cntry'].append(ISO[CNTRY_NM[times]])

	except:

		sqldict['Cntry'].append('')

		print(f'ISO Code for {CNTRY_NM[times]} NOT FOUND. Assigning empty cell to row {times+1}.')

	sqldict['InternalID_1_type'].append('#ID_RSSD')

	sqldict['InternalID_2_type'].append('ID_FDIC_CERT')

	sqldict['InternalID_3_type'].append('NM_SRCH_CD')

	sqldict['RegulationType'].append('Regulated')

	sqldict['Zip'].append(STATE_ABBR_NM[times]+' '+ZIP_CD[times])

	sqldict['ListProcessDate'].append(now.strftime('%Y-%m-%d'))

	sqldict['RegCtry'].append('US')

	sqldict['RegCode'].append('FFIEC')

	sqldict['ListCode'].append(regdict[PRIM_FED_REG[times]])

sqldict = bourange_same_length_array(sqldict)

###SQL ready files dict finishes###





#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

# normalize ListCode
df["ListCode"] = pd.to_numeric(df["ListCode"].astype(str).str.strip(), errors="coerce")
df["ListName"] = df["ListCode"].map(mapping)

df.to_excel(filename, index=False)

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


for tf in os.listdir(tempfolder):

	try:

		os.remove(tf)

	except:

		print(f'ERROR trying to delete {tf}, please delete manually...')

    

Running US FFIEC Web Scraping Tool v.1.1
ISO Code for NORTHERN IRELAND NOT FOUND. Assigning empty cell to row 51915.
ERROR trying to delete CSV_ATTRIBUTES_ACTIVE.CSV, please delete manually...
